# SMART AND common-split benchmark — FINAL clean version

This notebook creates a **strategy-level benchmark** for victim–perpetrator overlap detection under a **single common train/test split**.

Safeguards:

1. Overlap is defined count-wise as `V.SUM.TOTAL >= 1 AND P.SUM.TOTAL >= 1`.
2. The common split is stratified by the four-category role profile: neither, victimization only, perpetration only, overlap.
3. Scaler and PCA are fitted on the training partition only.
4. The direct overlap classifier and Smart AND benchmark are evaluated on the same adolescents.
5. This is not the direct combination of separately optimized saved final models; it is a strategy-level common-split benchmark.

Outputs are written to `final_smart_and_common_split_PCA_trainonly/`.


In [ ]:

# ============================================================
# 0. Configuration and imports
# ============================================================

from pathlib import Path
import os, json, random, warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, accuracy_score, balanced_accuracy_score,
    average_precision_score, roc_auc_score, classification_report
)
import joblib

RANDOM_STATE = 42
TEST_SIZE = 0.25
PCA_VARIANCE_THRESHOLD = 0.95
USE_FIXED_FINAL_COMPONENTS = True
FIXED_N_COMPONENTS = {"victimization": 18, "perpetration": 22, "overlap": 18}

VICTIM_THRESHOLD = 0.50
PERP_THRESHOLD = 0.50
DIRECT_OVERLAP_THRESHOLD = 0.50
OVERLAP_POS_WEIGHT_MULTIPLIER = 1.5

VICTIM_INNER_VAL_SIZE = 0.20
VICTIM_MIN_RECALL_FOR_ALPHA = 0.85

NN_EPOCHS = 200
NN_BATCH_SIZE = 128
NN_PATIENCE = 20
NN_VALIDATION_SPLIT = 0.15
NN_LEARNING_RATE = 1e-3

VICTIM_GRID = np.round(np.arange(0.05, 0.951, 0.005), 3)
PERP_GRID = np.round(np.arange(0.05, 0.951, 0.005), 3)

BASE_DIR = Path.cwd().resolve()
while BASE_DIR.name != "Version_Final" and BASE_DIR.parent != BASE_DIR:
    BASE_DIR = BASE_DIR.parent
if BASE_DIR.name != "Version_Final":
    BASE_DIR = Path.cwd().resolve()

DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "final_smart_and_common_split_PCA_trainonly"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
warnings.filterwarnings("ignore")

try:
    import tensorflow as tf
    from tensorflow.keras import layers, models
    from tensorflow.keras.callbacks import EarlyStopping
    TF_AVAILABLE = True
    try:
        tf.keras.mixed_precision.set_global_policy("float32")
    except Exception:
        pass
    tf.random.set_seed(RANDOM_STATE)
    print("TensorFlow available:", tf.__version__)
except Exception as e:
    TF_AVAILABLE = False
    print("TensorFlow not available:", e)

print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:

# ============================================================
# 1. Load analytical features and count-based targets
# ============================================================

V_COL = "V.SUM.TOTAL"
P_COL = "P.SUM.TOTAL"

LEAKAGE_COLS = [
    "VÍCTIMA", "VICTIMA", "PERPETRADOR", "VICTIMA_PERPETRADOR",
    "POLIVICTIMIZACION", "POLIPERPETRACION", "SOLO.VICTIMA",
    "SOLO.PERPETRADOR", "NO.VICT_NO.PERP", "V.O", P_COL, V_COL
]


def first_existing(candidates, required=True):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    if required:
        raise FileNotFoundError("None of these paths exist:\n" + "\n".join(map(str, candidates)))
    return None


def load_analytical_features_and_counts():
    feature_path = first_existing([
        DATA_DIR / "df_victima_feat.csv", BASE_DIR / "df_victima_feat.csv",
        DATA_DIR / "df_perpetrador_feat.csv", BASE_DIR / "df_perpetrador_feat.csv",
        DATA_DIR / "lista_global_vars.csv", BASE_DIR / "lista_global_vars.csv",
    ])
    mapping_path = first_existing([
        DATA_DIR / "analytical_sample_item_counts_mapping.csv",
        BASE_DIR / "analytical_sample_item_counts_mapping.csv",
    ], required=False)
    target_path = first_existing([DATA_DIR / "target_col.csv", BASE_DIR / "target_col.csv"])

    print("Feature path:", feature_path)
    print("Mapping path:", mapping_path)
    print("Target path:", target_path)

    feat = pd.read_csv(feature_path)
    target_raw = pd.read_csv(target_path)

    index_cols = [c for c in feat.columns if str(c).lower().startswith("unnamed") or str(c).lower() in ["index", "idx", "filtered_idx"]]
    if index_cols:
        idx_col = index_cols[0]
        feat_index = pd.to_numeric(feat[idx_col], errors="coerce").astype(int)
        feat = feat.drop(columns=[idx_col])
        feat.index = feat_index
    else:
        feat.index = np.arange(len(feat))

    if mapping_path is not None:
        mapping = pd.read_csv(mapping_path).copy()
        if "filtered_idx" not in mapping.columns:
            mapping["filtered_idx"] = np.arange(len(mapping))
        if "original_idx" not in mapping.columns:
            mapping["original_idx"] = mapping["filtered_idx"]
        mapping["filtered_idx"] = mapping["filtered_idx"].astype(int)
        mapping["original_idx"] = mapping["original_idx"].astype(int)
        mapping = mapping.set_index("filtered_idx", drop=False)

        if len(feat) != len(mapping):
            if len(feat) == len(target_raw) and mapping["original_idx"].max() < len(feat):
                feat = feat.iloc[mapping["original_idx"].to_numpy()].copy()
                feat.index = mapping.index
            else:
                raise RuntimeError(f"Feature rows ({len(feat)}) do not match mapping rows ({len(mapping)}).")
        else:
            feat = feat.copy()
            feat.index = mapping.index

        if {"victim_count", "perp_count"}.issubset(mapping.columns):
            counts = mapping[["filtered_idx", "original_idx", "victim_count", "perp_count"]].copy()
            counts[V_COL] = pd.to_numeric(counts["victim_count"], errors="coerce")
            counts[P_COL] = pd.to_numeric(counts["perp_count"], errors="coerce")
        else:
            tmp = target_raw.loc[mapping["original_idx"], [V_COL, P_COL]].copy()
            tmp.index = mapping.index
            counts = mapping[["filtered_idx", "original_idx"]].copy()
            counts[V_COL] = pd.to_numeric(tmp[V_COL], errors="coerce")
            counts[P_COL] = pd.to_numeric(tmp[P_COL], errors="coerce")
        counts.index = mapping.index
    else:
        print("WARNING: analytical mapping not found. Falling back to direct alignment.")
        if len(feat) != len(target_raw):
            raise RuntimeError("Mapping missing and feature/target lengths differ. Cannot safely reconstruct analytical sample.")
        counts = pd.DataFrame({
            "filtered_idx": feat.index.astype(int),
            "original_idx": feat.index.astype(int),
            V_COL: pd.to_numeric(target_raw[V_COL], errors="coerce").to_numpy(),
            P_COL: pd.to_numeric(target_raw[P_COL], errors="coerce").to_numpy(),
        }, index=feat.index)

    feat = feat.drop(columns=[c for c in LEAKAGE_COLS if c in feat.columns], errors="ignore").copy()
    for c in feat.columns:
        if feat[c].dtype == bool:
            feat[c] = feat[c].astype(int)
    object_cols = feat.select_dtypes(include=["object", "category"]).columns.tolist()
    if object_cols:
        print("Dummy-coding object/category columns:", object_cols)
        feat = pd.get_dummies(feat, columns=object_cols, drop_first=False)
    feat = feat.apply(pd.to_numeric, errors="raise")

    constant_cols = [c for c in feat.columns if feat[c].nunique(dropna=False) <= 1]
    if constant_cols:
        print("Dropping constant columns:", constant_cols)
        feat = feat.drop(columns=constant_cols)

    y_victim = (pd.to_numeric(counts[V_COL], errors="coerce") >= 1).astype(int)
    y_perp = (pd.to_numeric(counts[P_COL], errors="coerce") >= 1).astype(int)
    y_overlap = ((pd.to_numeric(counts[V_COL], errors="coerce") >= 1) & (pd.to_numeric(counts[P_COL], errors="coerce") >= 1)).astype(int)

    feat = feat.sort_index()
    counts = counts.loc[feat.index].copy()
    y_victim = y_victim.loc[feat.index]
    y_perp = y_perp.loc[feat.index]
    y_overlap = y_overlap.loc[feat.index]

    print("Analytical X:", feat.shape)
    print("Victim positives:", int(y_victim.sum()), "/", len(y_victim))
    print("Perp positives:", int(y_perp.sum()), "/", len(y_perp))
    print("Overlap positives:", int(y_overlap.sum()), "/", len(y_overlap))

    return feat, counts, y_victim, y_perp, y_overlap


X, counts_meta, y_victim, y_perp, y_overlap = load_analytical_features_and_counts()

X.to_csv(OUTPUT_DIR / "X_analytical_features_used.csv", index=True)
pd.DataFrame({
    "filtered_idx": X.index,
    "original_idx": counts_meta["original_idx"].to_numpy(),
    "y_victim": y_victim.to_numpy(),
    "y_perpetration": y_perp.to_numpy(),
    "y_overlap_count_based": y_overlap.to_numpy(),
    V_COL: counts_meta[V_COL].to_numpy(),
    P_COL: counts_meta[P_COL].to_numpy(),
}).to_csv(OUTPUT_DIR / "targets_count_based_used.csv", index=False)


In [ ]:

# ============================================================
# 2. Common four-category split
# ============================================================

joint_profile = (y_victim.astype(str) + "_" + y_perp.astype(str)).rename("joint_victim_perp_profile")
profile_table = joint_profile.value_counts().sort_index().rename_axis("profile").reset_index(name="n")
profile_table["percent"] = 100 * profile_table["n"] / len(joint_profile)
display(profile_table)
profile_table.to_csv(OUTPUT_DIR / "joint_profile_distribution_full_sample.csv", index=False)

idx_train, idx_test = train_test_split(
    X.index,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=joint_profile,
)
idx_train = pd.Index(idx_train).sort_values()
idx_test = pd.Index(idx_test).sort_values()

print("Train n:", len(idx_train))
print("Test n:", len(idx_test))
print("Train profile")
display(pd.crosstab(y_victim.loc[idx_train], y_perp.loc[idx_train], rownames=["Victim"], colnames=["Perp"], margins=True))
print("Test profile")
display(pd.crosstab(y_victim.loc[idx_test], y_perp.loc[idx_test], rownames=["Victim"], colnames=["Perp"], margins=True))

split_df = pd.DataFrame({
    "filtered_idx": list(idx_train) + list(idx_test),
    "split": ["train"] * len(idx_train) + ["test"] * len(idx_test),
})
split_df = split_df.merge(counts_meta.reset_index(drop=True)[["filtered_idx", "original_idx", V_COL, P_COL]], on="filtered_idx", how="left")
split_df["y_victim"] = split_df["filtered_idx"].map(y_victim)
split_df["y_perpetration"] = split_df["filtered_idx"].map(y_perp)
split_df["y_overlap"] = split_df["filtered_idx"].map(y_overlap)
split_df.to_csv(OUTPUT_DIR / "common_split_indices.csv", index=False)


In [ ]:

# ============================================================
# 3. Metrics, PCA, and model helpers
# ============================================================

def evaluate_binary(y_true, y_pred, y_prob=None, label=None):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    ppv = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan
    f1 = 2 * ppv * recall / (ppv + recall) if (ppv + recall) > 0 else np.nan
    row = {
        "label": label,
        "n": int(len(y_true)),
        "positive_support": int(np.sum(y_true == 1)),
        "negative_support": int(np.sum(y_true == 0)),
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
        "recall_sensitivity": recall,
        "specificity": specificity,
        "precision_ppv": ppv,
        "npv": npv,
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1_positive": f1,
        "accuracy": accuracy_score(y_true, y_pred),
    }
    if y_prob is not None:
        try:
            row["average_precision_pr_auc"] = average_precision_score(y_true, y_prob)
        except Exception:
            row["average_precision_pr_auc"] = np.nan
        try:
            row["roc_auc"] = roc_auc_score(y_true, y_prob)
        except Exception:
            row["roc_auc"] = np.nan
    return row


def pct_table(df):
    out = df.copy()
    for c in ["recall_sensitivity", "specificity", "precision_ppv", "npv", "balanced_accuracy", "f1_positive", "accuracy", "average_precision_pr_auc", "roc_auc"]:
        if c in out.columns:
            out[c] = out[c].map(lambda x: f"{x*100:.1f}%" if pd.notna(x) else "")
    return out


def fit_pca_train_only(X, train_idx, test_idx, outcome):
    X_train_raw = X.loc[train_idx].copy()
    X_test_raw = X.loc[test_idx].copy()
    scaler = MinMaxScaler(feature_range=(0, 1))
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_test_scaled = scaler.transform(X_test_raw)
    train_means = X_train_scaled.mean(axis=0)
    X_train_centered = X_train_scaled - train_means
    X_test_centered = X_test_scaled - train_means
    pca_full = PCA(n_components=X_train_raw.shape[1], random_state=RANDOM_STATE)
    X_train_pca_full = pca_full.fit_transform(X_train_centered)
    X_test_pca_full = pca_full.transform(X_test_centered)
    n_components = FIXED_N_COMPONENTS[outcome] if USE_FIXED_FINAL_COMPONENTS else int(np.searchsorted(np.cumsum(pca_full.explained_variance_ratio_), PCA_VARIANCE_THRESHOLD) + 1)
    pc_cols = [f"PC{i+1}" for i in range(n_components)]
    X_train_pca = pd.DataFrame(X_train_pca_full[:, :n_components], index=train_idx, columns=pc_cols)
    X_test_pca = pd.DataFrame(X_test_pca_full[:, :n_components], index=test_idx, columns=pc_cols)
    variance_df = pd.DataFrame({
        "outcome": outcome,
        "component_number": np.arange(1, len(pca_full.explained_variance_ratio_) + 1),
        "component": [f"PC{i}" for i in range(1, len(pca_full.explained_variance_ratio_) + 1)],
        "explained_variance_ratio": pca_full.explained_variance_ratio_,
        "explained_variance_percent": pca_full.explained_variance_ratio_ * 100,
        "cumulative_variance_ratio": np.cumsum(pca_full.explained_variance_ratio_),
        "cumulative_variance_percent": np.cumsum(pca_full.explained_variance_ratio_) * 100,
        "retained": np.arange(1, len(pca_full.explained_variance_ratio_) + 1) <= n_components,
    })
    meta = {
        "scaler": scaler,
        "train_means": train_means,
        "pca_full": pca_full,
        "n_components": n_components,
        "feature_names": list(X.columns),
        "variance_df": variance_df,
        "retained_variance_percent": float(variance_df.loc[variance_df["retained"], "explained_variance_percent"].sum()),
    }
    return X_train_pca, X_test_pca, meta


def select_victim_tree_alpha(X_train_pca, y_train):
    inner_train_idx, inner_val_idx = train_test_split(
        X_train_pca.index,
        test_size=VICTIM_INNER_VAL_SIZE,
        random_state=RANDOM_STATE,
        stratify=y_train.loc[X_train_pca.index],
    )
    inner_train_idx = pd.Index(inner_train_idx)
    inner_val_idx = pd.Index(inner_val_idx)
    full_tree = DecisionTreeClassifier(random_state=RANDOM_STATE)
    full_tree.fit(X_train_pca.loc[inner_train_idx], y_train.loc[inner_train_idx])
    path = full_tree.cost_complexity_pruning_path(X_train_pca.loc[inner_train_idx], y_train.loc[inner_train_idx])
    rows = []
    for alpha in np.unique(path.ccp_alphas):
        clf = DecisionTreeClassifier(random_state=RANDOM_STATE, ccp_alpha=float(alpha))
        clf.fit(X_train_pca.loc[inner_train_idx], y_train.loc[inner_train_idx])
        prob = clf.predict_proba(X_train_pca.loc[inner_val_idx])[:, 1]
        pred = (prob >= VICTIM_THRESHOLD).astype(int)
        m = evaluate_binary(y_train.loc[inner_val_idx], pred, prob, label="victim_inner_val")
        m.update({"ccp_alpha": float(alpha), "depth": clf.get_depth(), "n_leaves": clf.get_n_leaves()})
        rows.append(m)
    alpha_df = pd.DataFrame(rows)
    candidates = alpha_df[alpha_df["recall_sensitivity"] >= VICTIM_MIN_RECALL_FOR_ALPHA].copy()
    if len(candidates) == 0:
        selected = alpha_df.sort_values(["recall_sensitivity", "balanced_accuracy", "specificity", "precision_ppv"], ascending=False).iloc[0]
    else:
        selected = candidates.sort_values(["balanced_accuracy", "specificity", "precision_ppv", "n_leaves"], ascending=[False, False, False, True]).iloc[0]
    return float(selected["ccp_alpha"]), alpha_df, selected


def train_victim_tree(X_train_pca, X_test_pca, y_train):
    alpha, alpha_df, selected = select_victim_tree_alpha(X_train_pca, y_train)
    model = DecisionTreeClassifier(random_state=RANDOM_STATE, ccp_alpha=alpha)
    model.fit(X_train_pca, y_train.loc[X_train_pca.index])
    prob = model.predict_proba(X_test_pca)[:, 1]
    pred = (prob >= VICTIM_THRESHOLD).astype(int)
    return model, prob, pred, alpha_df, selected


def build_perp_dnn(input_dim):
    if not TF_AVAILABLE:
        raise ImportError("TensorFlow is required for the DNN perpetration benchmark model.")
    tf.keras.backend.clear_session()
    tf.random.set_seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)
    random.seed(RANDOM_STATE)
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.20),
        layers.Dense(16, activation="relu"),
        layers.Dropout(0.20),
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=NN_LEARNING_RATE),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.Recall(name="recall"), tf.keras.metrics.Precision(name="precision"), tf.keras.metrics.BinaryAccuracy(name="accuracy")],
    )
    return model


def train_perp_dnn(X_train_pca, X_test_pca, y_train, verbose=0):
    ytr = y_train.loc[X_train_pca.index].astype(int).to_numpy()
    neg = int(np.sum(ytr == 0)); pos = int(np.sum(ytr == 1)); total = neg + pos
    class_weight = {0: total / (2 * neg), 1: total / (2 * pos)}
    model = build_perp_dnn(X_train_pca.shape[1])
    history = model.fit(
        X_train_pca.values, ytr,
        validation_split=NN_VALIDATION_SPLIT,
        epochs=NN_EPOCHS,
        batch_size=NN_BATCH_SIZE,
        class_weight=class_weight,
        callbacks=[EarlyStopping(monitor="val_recall", mode="max", patience=NN_PATIENCE, restore_best_weights=True)],
        verbose=verbose,
    )
    prob = model.predict(X_test_pca.values, verbose=0).ravel()
    pred = (prob >= PERP_THRESHOLD).astype(int)
    return model, prob, pred, history, class_weight


def train_direct_overlap_logreg(X_train_pca, X_test_pca, y_train):
    ytr = y_train.loc[X_train_pca.index].astype(int).to_numpy()
    pos_rate = ytr.mean()
    base_pos_weight = (1 - pos_rate) / pos_rate
    sample_weight = np.where(ytr == 1, base_pos_weight * OVERLAP_POS_WEIGHT_MULTIPLIER, 1.0)
    model = LogisticRegression(max_iter=5000, solver="liblinear", random_state=RANDOM_STATE)
    model.fit(X_train_pca.values, ytr, sample_weight=sample_weight)
    prob = model.predict_proba(X_test_pca.values)[:, 1]
    pred = (prob >= DIRECT_OVERLAP_THRESHOLD).astype(int)
    config = {
        "pos_rate_train": float(pos_rate),
        "base_pos_weight": float(base_pos_weight),
        "pos_weight_multiplier": float(OVERLAP_POS_WEIGHT_MULTIPLIER),
        "positive_class_final_weight": float(base_pos_weight * OVERLAP_POS_WEIGHT_MULTIPLIER),
    }
    return model, prob, pred, config


In [ ]:

# ============================================================
# 4. Fit PCA pipelines and train benchmark models
# ============================================================

Xv_train_pca, Xv_test_pca, victim_pca_meta = fit_pca_train_only(X, idx_train, idx_test, "victimization")
Xp_train_pca, Xp_test_pca, perp_pca_meta = fit_pca_train_only(X, idx_train, idx_test, "perpetration")
Xo_train_pca, Xo_test_pca, overlap_pca_meta = fit_pca_train_only(X, idx_train, idx_test, "overlap")

print("Victim PCA:", Xv_train_pca.shape, "retained variance", victim_pca_meta["retained_variance_percent"])
print("Perp PCA:", Xp_train_pca.shape, "retained variance", perp_pca_meta["retained_variance_percent"])
print("Overlap PCA:", Xo_train_pca.shape, "retained variance", overlap_pca_meta["retained_variance_percent"])

victim_model, victim_prob, victim_pred, victim_alpha_df, victim_alpha_selected = train_victim_tree(
    Xv_train_pca, Xv_test_pca, y_victim.loc[idx_train]
)
print("Victim selected alpha:", float(victim_alpha_selected["ccp_alpha"]))
print("Victim tree depth/leaves:", victim_model.get_depth(), victim_model.get_n_leaves())
print(classification_report(y_victim.loc[idx_test], victim_pred, digits=3))

perp_model, perp_prob, perp_pred, perp_history, perp_class_weight = train_perp_dnn(
    Xp_train_pca, Xp_test_pca, y_perp.loc[idx_train], verbose=0
)
print("Perp class_weight:", perp_class_weight)
print(classification_report(y_perp.loc[idx_test], perp_pred, digits=3))

ol_model, ol_prob, ol_pred, ol_config = train_direct_overlap_logreg(
    Xo_train_pca, Xo_test_pca, y_overlap.loc[idx_train]
)
print("Direct overlap logistic config:", ol_config)
print(classification_report(y_overlap.loc[idx_test], ol_pred, digits=3))


In [ ]:

# ============================================================
# 5. Smart AND strategies
# ============================================================

smart_and_default_pred = ((victim_pred == 1) & (perp_pred == 1)).astype(int)
smart_and_default_prob = np.minimum(victim_prob, perp_prob)


def smart_and_grid(y_true_overlap, victim_prob, perp_prob, target_recall):
    rows = []
    y_true_overlap = np.asarray(y_true_overlap).astype(int)
    min_prob = np.minimum(victim_prob, perp_prob)
    for vt in VICTIM_GRID:
        v_pred = victim_prob >= vt
        for pt in PERP_GRID:
            p_pred = perp_prob >= pt
            pred = (v_pred & p_pred).astype(int)
            m = evaluate_binary(y_true_overlap, pred, min_prob, label="Smart AND grid")
            m.update({
                "victim_threshold": float(vt),
                "perp_threshold": float(pt),
                "recall_gap_vs_direct": abs(m["recall_sensitivity"] - target_recall),
            })
            rows.append(m)
    return pd.DataFrame(rows)


direct_overlap_metrics = evaluate_binary(y_overlap.loc[idx_test], ol_pred, ol_prob, label="Direct overlap classifier")
target_recall = direct_overlap_metrics["recall_sensitivity"]
print("Direct overlap test recall used for recall-matched benchmark:", target_recall)

grid_df = smart_and_grid(y_overlap.loc[idx_test], victim_prob, perp_prob, target_recall=target_recall)

eligible = grid_df[grid_df["recall_sensitivity"] >= target_recall].copy()
if len(eligible) == 0:
    eligible = grid_df.copy()
    selection_note = "No Smart AND threshold pair reached direct-overlap recall; selected closest recall."
else:
    selection_note = "Selected Smart AND threshold pair with recall >= direct-overlap recall and best false-positive control."

smart_selected = eligible.sort_values(
    ["recall_gap_vs_direct", "specificity", "balanced_accuracy", "precision_ppv"],
    ascending=[True, False, False, False],
).iloc[0]

SMART_VICTIM_THRESHOLD = float(smart_selected["victim_threshold"])
SMART_PERP_THRESHOLD = float(smart_selected["perp_threshold"])
smart_and_matched_pred = ((victim_prob >= SMART_VICTIM_THRESHOLD) & (perp_prob >= SMART_PERP_THRESHOLD)).astype(int)
smart_and_matched_prob = np.minimum(victim_prob, perp_prob)

print("Smart AND selected thresholds:", SMART_VICTIM_THRESHOLD, SMART_PERP_THRESHOLD)
print("selection_note:", selection_note)
display(pd.DataFrame([smart_selected]))

print("Smart AND default")
print(classification_report(y_overlap.loc[idx_test], smart_and_default_pred, digits=3))
print("Smart AND recall-matched")
print(classification_report(y_overlap.loc[idx_test], smart_and_matched_pred, digits=3))


In [ ]:

# ============================================================
# 6. Final benchmark tables and outputs
# ============================================================

metrics_rows = []
metrics_rows.append({
    "strategy": "Victimization role classifier",
    "model": "Cost-complexity pruned decision tree",
    "threshold_type": "role model",
    "victim_threshold": VICTIM_THRESHOLD,
    "perp_threshold": np.nan,
    "overlap_threshold": np.nan,
    **evaluate_binary(y_victim.loc[idx_test], victim_pred, victim_prob, label="victimization_role")
})
metrics_rows.append({
    "strategy": "Perpetration role classifier",
    "model": "Compact feed-forward neural network",
    "threshold_type": "role model",
    "victim_threshold": np.nan,
    "perp_threshold": PERP_THRESHOLD,
    "overlap_threshold": np.nan,
    **evaluate_binary(y_perp.loc[idx_test], perp_pred, perp_prob, label="perpetration_role")
})
metrics_rows.append({
    "strategy": "Direct overlap classifier",
    "model": "Weighted logistic regression SW_pos1.5",
    "threshold_type": "direct overlap threshold",
    "victim_threshold": np.nan,
    "perp_threshold": np.nan,
    "overlap_threshold": DIRECT_OVERLAP_THRESHOLD,
    **evaluate_binary(y_overlap.loc[idx_test], ol_pred, ol_prob, label="direct_overlap")
})
metrics_rows.append({
    "strategy": "Smart AND default",
    "model": "Victimization prediction AND perpetration prediction",
    "threshold_type": "default role thresholds",
    "victim_threshold": VICTIM_THRESHOLD,
    "perp_threshold": PERP_THRESHOLD,
    "overlap_threshold": np.nan,
    **evaluate_binary(y_overlap.loc[idx_test], smart_and_default_pred, smart_and_default_prob, label="smart_and_default")
})
metrics_rows.append({
    "strategy": "Smart AND recall-matched benchmark",
    "model": "Victimization prediction AND perpetration prediction",
    "threshold_type": "recall-matched benchmark thresholds",
    "victim_threshold": SMART_VICTIM_THRESHOLD,
    "perp_threshold": SMART_PERP_THRESHOLD,
    "overlap_threshold": np.nan,
    **evaluate_binary(y_overlap.loc[idx_test], smart_and_matched_pred, smart_and_matched_prob, label="smart_and_recall_matched")
})

metrics_df = pd.DataFrame(metrics_rows)
metrics_pct = pct_table(metrics_df)
metrics_df.to_csv(OUTPUT_DIR / "strategy_benchmark_common_split_metrics.csv", index=False)
metrics_pct.to_csv(OUTPUT_DIR / "strategy_benchmark_common_split_metrics_pretty.csv", index=False)
grid_df.to_csv(OUTPUT_DIR / "smart_and_threshold_grid_common_test.csv", index=False)

display(metrics_pct)

supp_table = metrics_df[metrics_df["strategy"].isin(["Direct overlap classifier", "Smart AND recall-matched benchmark"])].copy()
supp_table_pretty = pct_table(supp_table)
supp_table.to_csv(OUTPUT_DIR / "supplement_direct_overlap_vs_smart_and.csv", index=False)
supp_table_pretty.to_csv(OUTPUT_DIR / "supplement_direct_overlap_vs_smart_and_pretty.csv", index=False)
display(supp_table_pretty)


In [ ]:

# ============================================================
# 7. Save row-level predictions and artifacts
# ============================================================

test_meta = counts_meta.loc[idx_test].copy()
pred_df = pd.DataFrame({
    "filtered_idx": idx_test,
    "original_idx": test_meta["original_idx"].to_numpy(),
    V_COL: test_meta[V_COL].to_numpy(),
    P_COL: test_meta[P_COL].to_numpy(),
    "y_true_victim": y_victim.loc[idx_test].to_numpy().astype(int),
    "y_true_perpetration": y_perp.loc[idx_test].to_numpy().astype(int),
    "y_true_overlap": y_overlap.loc[idx_test].to_numpy().astype(int),
    "y_prob_victim_role": victim_prob,
    "y_pred_victim_role_default": victim_pred,
    "y_prob_perp_role": perp_prob,
    "y_pred_perp_role_default": perp_pred,
    "y_prob_direct_overlap": ol_prob,
    "y_pred_direct_overlap": ol_pred,
    "y_prob_smart_and_min": smart_and_default_prob,
    "y_pred_smart_and_default": smart_and_default_pred,
    "y_pred_smart_and_recall_matched": smart_and_matched_pred,
})
pred_df.to_csv(OUTPUT_DIR / "common_split_test_predictions_all_strategies.csv", index=False)

for outcome, meta in [("victimization", victim_pca_meta), ("perpetration", perp_pca_meta), ("overlap", overlap_pca_meta)]:
    meta["variance_df"].to_csv(OUTPUT_DIR / f"pca_variance_{outcome}.csv", index=False)
    joblib.dump(meta["scaler"], OUTPUT_DIR / f"scaler_{outcome}.joblib")
    joblib.dump(meta["pca_full"], OUTPUT_DIR / f"pca_full_{outcome}.joblib")
    pd.DataFrame({"feature": meta["feature_names"]}).to_csv(OUTPUT_DIR / f"pca_input_features_{outcome}.csv", index=False)

joblib.dump(victim_model, OUTPUT_DIR / "victim_pruned_tree_common_split.joblib")
joblib.dump(ol_model, OUTPUT_DIR / "direct_overlap_logreg_SW_pos1p5_common_split.joblib")
try:
    perp_model.save(OUTPUT_DIR / "perpetration_dnn_common_split.keras")
except Exception as e:
    print("Could not save DNN model:", e)

victim_alpha_df.to_csv(OUTPUT_DIR / "victim_tree_alpha_selection_inner_validation.csv", index=False)
pd.DataFrame(perp_history.history).to_csv(OUTPUT_DIR / "perpetration_dnn_history.csv", index=False)

config = {
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "split": "common 75/25 split stratified by four-category victimization/perpetration role profile",
    "outcome_definitions": {
        "victimization": "V.SUM.TOTAL >= 1",
        "perpetration": "P.SUM.TOTAL >= 1",
        "overlap": "V.SUM.TOTAL >= 1 AND P.SUM.TOTAL >= 1",
    },
    "pca": {
        "fit": "training partition only",
        "use_fixed_final_components": USE_FIXED_FINAL_COMPONENTS,
        "fixed_n_components": FIXED_N_COMPONENTS,
        "variance_threshold": PCA_VARIANCE_THRESHOLD,
        "victimization_retained_variance_percent": victim_pca_meta["retained_variance_percent"],
        "perpetration_retained_variance_percent": perp_pca_meta["retained_variance_percent"],
        "overlap_retained_variance_percent": overlap_pca_meta["retained_variance_percent"],
    },
    "models": {
        "victimization_role": "cost-complexity pruned decision tree, ccp_alpha selected by inner validation within common training set",
        "perpetration_role": "compact feed-forward neural network with class weights and early stopping on validation recall",
        "direct_overlap": "weighted logistic regression SW_pos1.5",
        "smart_and": "positive only when both role-specific predictions are positive",
    },
    "thresholds": {
        "victim_default": VICTIM_THRESHOLD,
        "perp_default": PERP_THRESHOLD,
        "direct_overlap": DIRECT_OVERLAP_THRESHOLD,
        "smart_and_recall_matched_victim": SMART_VICTIM_THRESHOLD,
        "smart_and_recall_matched_perp": SMART_PERP_THRESHOLD,
        "smart_and_selection_note": selection_note,
    },
    "interpretation_note": "Supplementary strategy-level common-split benchmark; not the direct combination of separately optimized saved final classifiers.",
}
with open(OUTPUT_DIR / "smart_and_common_split_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

summary_lines = [
    "SMART AND common-split benchmark — FINAL clean version",
    "The benchmark used a common 75/25 split stratified by the four-category victimization/perpetration role profile.",
    "Overlap was defined count-wise as V.SUM.TOTAL >= 1 AND P.SUM.TOTAL >= 1.",
    "Scaler and PCA were fitted on the common training partition only.",
    "The Smart AND strategy classified adolescents as overlap-positive only when both role-specific predictions were positive.",
    "This is a strategy-level benchmark, not the direct combination of separately optimized saved final classifiers.",
    "",
    supp_table_pretty.to_string(index=False),
]
with open(OUTPUT_DIR / "README_for_manuscript.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines))

print("Saved outputs to:", OUTPUT_DIR)
for fname in [
    "strategy_benchmark_common_split_metrics.csv",
    "strategy_benchmark_common_split_metrics_pretty.csv",
    "supplement_direct_overlap_vs_smart_and_pretty.csv",
    "common_split_test_predictions_all_strategies.csv",
    "smart_and_threshold_grid_common_test.csv",
    "README_for_manuscript.txt",
]:
    print("-", OUTPUT_DIR / fname)


## Interpretation guidance

Use this benchmark only if the common-split comparison is clear and methodologically useful.

Suggested table note:

> Note. This supplementary benchmark used a common 75/25 split stratified by the four-category role profile. The Smart AND strategy classified an adolescent as overlap-positive only when both role-specific predictions were positive. This comparison is a strategy-level benchmark and should not be interpreted as the direct combination of the separately optimized saved final victimization and perpetration classifiers. Scaler and PCA transformations were fitted on the common training partition only.

If the direct overlap classifier is clearly superior at similar recall, suggested wording:

> A supplementary common-split benchmark supported treating victim–perpetrator overlap as a direct prediction target rather than only as the conjunction of separate victimization and perpetration predictions.

If results are mixed or unclear, leave the benchmark out of the manuscript.
